In [ ]:
from fefetlab.instruments.visa_session import VisaSession, VisaConfig
from fefetlab.b1500 import B1500

cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=10000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

with VisaSession(cfg) as s:
    b = B1500(s)

    print("*IDN? ->", s.query("*IDN?"))
    print("UNT?  ->", s.query("UNT?"))
    print("LOP?  ->", s.query("LOP?"))

    for t in range(1, 11):
        try:
            print(f"*LRN? {t} ->", s.query(f"*LRN? {t}"))
        except Exception as e:
            print(f"*LRN? {t} -> ERROR:", e)


In [ ]:
from fefetlab.instruments.visa_session import VisaSession, VisaConfig
from fefetlab.b1500 import B1500, B1500Error

cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=10000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

test_channels = [4, 5, 6, 7]

with VisaSession(cfg) as s:
    b = B1500(s)

    print("IDN:", s.query("*IDN?"))
    print("UNT?:", s.query("UNT?"))
    print("LOP?:", s.query("LOP?"))

    for ch in test_channels:
        print(f"\n=== Probe channel {ch} ===")
        print("ERRX drained at start:", b.clear_err_queue())
        try:
            b.fmt(5)
            print("FMT ok")

            b.cn([ch])
            print(f"CN {ch} ok")

            b.dv(ch, 0, 0.0, 1e-3)
            print(f"DV {ch} ok")
            print("ERRX after DV:", b.errx())
        except Exception as e:
            print(f"Channel {ch}: FAIL -> {e}")
        finally:
            try:
                b.dz([ch])
                print(f"DZ {ch} ok")
            except Exception as e:
                print(f"Channel {ch}: DZ fail -> {e}")

            try:
                b.cl([ch])
                print(f"CL {ch} ok")
            except Exception as e:
                print(f"Channel {ch}: CL fail -> {e}")

            print("ERRX drained at end:", b.clear_err_queue())
